In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [4]:
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
sequences = ["I've been waiting for a HuggingFace course my whole life.", "So have I!"]

tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")
output = model(**tokens)


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6583.37it/s]


In [5]:
model.eval()

with torch.inference_mode():
    baseline_logits = model(**tokens).logits.clone()

print("Baseline: ", baseline_logits)

Baseline:  tensor([[-1.5607,  1.6123],
        [-3.6183,  3.9137]])


In [27]:
def custom_activation(x):
    return torch.nn.functional.hardswish(x)

class CustomActivation(torch.nn.Module):
    def forward(self, x):
        return custom_activation(x)

In [28]:
original_activations = [
    block.ffn.activation
    for block in model.distilbert.transformer.layer
]

for block in model.distilbert.transformer.layer:
    block.ffn.activation = CustomActivation()


In [29]:
model.eval()

with torch.inference_mode():
    modified_logits = model(**tokens).logits

print("Baseline:", baseline_logits)
print("Modified:", modified_logits)

print(
    "Maximum absolute logit difference:",
    (modified_logits - baseline_logits).abs().max().item(),
)

print("Baseline classes:", baseline_logits.argmax(dim=-1))
print("Modified classes:", modified_logits.argmax(dim=-1))

Baseline: tensor([[-1.5607,  1.6123],
        [-3.6183,  3.9137]])
Modified: tensor([[ 2.1781, -1.7498],
        [ 0.7501, -0.4372]])
Maximum absolute logit difference: 4.368403434753418
Baseline classes: tensor([1, 1])
Modified classes: tensor([0, 0])
